# Script de python sencillo para descargar datos de EAGLE 

#### Antes de empezar: una herramienta útil para descargar datos de EAGLE desde Python, es el paquete eagleSQLtools.

#### Repositorio del paquete: https://github.com/kyleaoman/eagleSqlTools

#### Para instalarlo, en una terminal correr el siguiente comando:

#### pip install eaglesqltools

#### Importar paquetes necesarios. Agregar paquete si hace falta

In [1]:
import matplotlib.pyplot as plt
import eagleSqlTools as sql
from astropy.io import ascii
from astropy.table import Table
import numpy as np

#### Datos para la descarga (usuario, contraseña, simulación, snapnum, fragmentos de la query)

In [15]:
# Usuario y contraseña
usr='cht015'
pwd='BH457tfj'

# =========================================================================================

# Simulación y snapnum deseado
simu='NoAGNL0050N0752'
snap=28

# =========================================================================================

# Carpeta donde guardar archivo. Dejar un caracter 'vacío' para descargar los datos
# en la misma carpeta desde donde se ejecuta la notebook.
download_folder='/home/ramiro/Documentos/Facultad/Datos guardados/Maestría/Taller de Tesis 1/Python Taller 1/data/'

# Nombre del archivo que guardará los datos descargados
filename='MuestraMZR_'+simu+'_snap_'+str(snap)+'.dat'

# =========================================================================================
# Cosas para armar la query

# Tablas y Alias de las tablas desde donde quiero descargar datos
# Agregar/quitar según sea necesario
tables=[
        'Subhalo'
          ]

aliases=[
         'sub'

        ]

# --------------------------------------------------------------------------------

# Columnas a seleccionar de cada tabla (agregar/quitar según sea necesario)
columns=[ [
           'GalaxyID','GroupID','SnapNum','SubGroupNumber','Stars_Mass','BlackHoleMass','SF_Mass','SF_Oxygen','SF_Hydrogen'
          ]   
             ]

# --------------------------------------------------------------------------------

# Condiciones sobre las columnas de las tablas a seleccionar
# NO AGREGAR ACÁ EL ALIAS DE LA TABLA! Se agregará después
# Agregar/quitar según sea necesario
# Cada lista de la siguiente lista establece condiciones para la tabla
# a seleccionar (respetando el orden en que se definieron las tablas).
# Si no se quieren condiciones sobre alguna tabla, dejar la correspondiente
# lista como vacía.

conditions=[
            ['SnapNum='+str(snap),'BlackHoleMass>='+str(0),'Stars_Mass>'+str(1e10),'SF_Oxygen>'+str(0),'SF_Hydrogen>'+str(0)]
           ]


# --------------------------------------------------------------------------------


#### Construcción de las distintas partes de la query

In [16]:
# Armo la sentencia SELECT 
select=''
for alias,col in zip(aliases,columns):
    select=select+(','.join([alias+'.'+name for name in col]))+','
select=select[:-1]   # Esto es para borrar una última coma 'molesta'

# --------------------------------------------------------------------------------

# Armo la sentencia FROM
from_table=''
for tab,alias in zip(tables,aliases):
    from_table=from_table+simu+'_'+tab+' as '+alias+','
from_table=from_table[:-1]   # Esto es para borrar una última coma 'molesta'

# --------------------------------------------------------------------------------

# Vemos las condiciones para el WHERE.
for cond in conditions:
    print(cond, type(cond))


# Armo la sentencia WHERE
# Condiciones para unir tablas (igualdad de GalaxyIDs)
join_conditions=''
# Lo sigueinte solo se ejecutará si hay más de una tabla
if len(tables)>1:
    for k in range(len(aliases)-1):
        join_conditions=(join_conditions+
                         aliases[k]+'.GalaxyID='+aliases[k+1]+'.GalaxyID'+' and ')
    join_conditions=join_conditions[:-5]

where=''    
for alias,condit in zip(aliases,conditions):
    if len(condit)>0:
        where=where+(' and '.join(alias+'.'+cond for cond in condit))+ ' and '
        if len(aliases)==1:
            where=where[:-4] # Esto es para borrar un 'and' molesto
where=where+join_conditions

['SnapNum=28', 'BlackHoleMass>=0', 'Stars_Mass>10000000000.0', 'SF_Oxygen>0', 'SF_Hydrogen>0'] <class 'list'>


In [17]:
# Testeo de la query
print('SELECT')
print(select)
print()
print('FROM')
print(from_table)
print()
print('WHERE')
print(where)

SELECT
sub.GalaxyID,sub.GroupID,sub.SnapNum,sub.SubGroupNumber,sub.Stars_Mass,sub.BlackHoleMass,sub.SF_Mass,sub.SF_Oxygen,sub.SF_Hydrogen

FROM
NoAGNL0050N0752_Subhalo as sub

WHERE
sub.SnapNum=28 and sub.BlackHoleMass>=0 and sub.Stars_Mass>10000000000.0 and sub.SF_Oxygen>0 and sub.SF_Hydrogen>0 


#### Conexión a la base de datos y descarga

In [18]:
# Conectarse a la base de datos
con = sql.connect(usr,password=pwd)

# Query en SQL
query = ' SELECT '+select+' FROM '+from_table +' WHERE '+where

# Execute query 
exquery = sql.execute_query(con, query)

# List of column names of downloaded data
colnames=(exquery.view(np.recarray).dtype.names)

# Dictionary of data
mytable={}
for name in colnames:
    mytable[name]=exquery[name]
    
# dictionary={key1:value,key2:value,....}
    

#### Guardar los datos en un archivo ascii

In [19]:
# Abrir el archivo donde se guardarán los datos
outf = open(download_folder+filename,'w')

# Transformar el diccionario a una tabla ascii
data_ascii=Table(mytable)

# Escribir los datos al archivo
data_ascii.write(outf,format='csv')

# Cerrar el archivo
outf.close()

In [20]:
data_ascii

GalaxyID,GroupID,SnapNum,SubGroupNumber,Stars_Mass,BlackHoleMass,SF_Mass,SF_Oxygen,SF_Hydrogen
int64,int64,int32,int32,float32,float32,float32,float32,float32
1620592,28000000000014,28,0,2.7566093e+11,0.0,3.2629962e+09,0.033660218,0.55353934
1629768,28000000000014,28,2,1.5534852e+11,0.0,5.4378266e+09,0.03157206,0.5668449
1650810,28000000000014,28,1,1.4536788e+11,0.0,2.1184418e+10,0.0143695045,0.6737222
3126435,28000000000135,28,0,3.912528e+10,0.0,6.011917e+09,0.014139314,0.67576754
3131127,28000000000136,28,0,1.3175951e+10,0.0,1.7318542e+09,0.014267102,0.6746438
3136043,28000000000137,28,0,4.2697314e+10,0.0,7.836248e+09,0.01632721,0.66447306
3139793,28000000000138,28,0,4.4419617e+10,0.0,1.1539818e+10,0.01446732,0.67664415
3143795,28000000000139,28,0,5.220936e+10,0.0,6.9703977e+09,0.016568411,0.66212696
3147527,28000000000140,28,0,6.7998654e+10,0.0,6.8181023e+09,0.02041306,0.6370468


#### EoP